# RAG Evaluation

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = "C:/Users/mamen/Documents/Python/RAGChatBot"

sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:/Users/mamen/Documents/Python/RAGChatBot


In [2]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

gold_df_path = f"{PROJECT_ROOT}/eval/gold_references.csv"
gold_df = pd.read_csv(gold_df_path)

gold_df.head()

,question_id,question,evidence_id,expected_doc,expected_page,expected_text,notes
0,Q001,¿Qué es la astenia?,Q001_E01,general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf,9,Astenia es el término médico para denominar al cansancio (falta de energía) ya sea físico o emocional.,NaN
1,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,Q002_E01,general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf,42,"Si fiebre de más de 38º C, hay que tomar un antipirético (paracetamol, ibuprofeno, metamizol… ) si tras 6-8 horas vuelve a subir la fi ebre, hay que acudir a urgencias para realizar las pruebas pertinentes.",NaN
2,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,Q002_E02,general_es_pfizer_manual-pacientes_2007.pdf,34,"Si tiene temperatura de 38º C o más, debe de ir a urgencias.\nSi tiene fiebre de 38º C o superior, debe avisar a su médico o ir al hospital.",NaN
3,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,Q002_E03,general_es_pfizer_manual-pacientes_2007.pdf,150,"Una temperatura igual o superior a 38ºC indica la posibilidad de una infección y requiere consultar al médico o enfermera qué se debe hacer en una situación así. Hay ocasiones en las que algún medicamento, o la propia enfermedad, pueden provocar fiebre.",NaN
4,Q003,¿Cuál es el trastorno del sueño más común en pacientes con cáncer?,Q003_E01,general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf,47,El insomnio es el trastorno más común en los pacientes con cáncer y suele ser secundario a factores físicos y psicológicos relacionados con el cáncer y sus tratamientos.,NaN


In [6]:
#save the gold_df to a json file in the eval folder.
gold_df.to_json(f"{PROJECT_ROOT}/eval/gold_references.json", orient="records", lines=True)

# Validation check

In [9]:
question_id = "Q005"


gold_df[gold_df["question_id"] == question_id]

,question_id,question,evidence_id,expected_doc,expected_page,expected_text,notes
6,Q005,¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?,Q005_E01,general_es_pfizer_manual-pacientes_2007.pdf,33,Suele durar entre 2 y 6 horas.,NaN


In [10]:
#Check which rows have the column "notes" not equal to NaN
gold_df[gold_df["notes"].notna()]

,question_id,question,evidence_id,expected_doc,expected_page,expected_text,notes


In [11]:
#Check which rows have NaN values in expected_doc, expected_page or expected_text columns
gold_df[gold_df[["expected_doc", "expected_page", "expected_text"]].isna().any(axis=1)]


,question_id,question,evidence_id,expected_doc,expected_page,expected_text,notes


In [9]:
#Prepare unique questions
questions_df = (
    gold_df[["question_id", "question"]]
    .drop_duplicates()
    .sort_values("question_id")
    .reset_index(drop=True)
)

print("Número de preguntas:", len(questions_df))
print("Número de evidencias:", len(gold_df))

Número de preguntas: 31
Número de evidencias: 77


# Evaluate Retrieval

3 types of retrieval are implemented: 
- Semantic search via function `search()`
- Hybrid search via function `hybrid_search()`: combining semantic (pgvector cosine) and keyword/full-text
- Language aware hybrid search via function `language_aware_hybrid_search()`: A wrapper around `hybrid_search` that adds a language filter to prioritize chunks in the user's language.

In [36]:
page_tolerance = 1
TOP_K = 5

## Auxiliary functions

In [10]:
def normalize_doc_name(path_or_name):
    """
    Convierte rutas tipo docs/mama/documento.pdf en documento.pdf.
    """
    if pd.isna(path_or_name):
        return ""
    return Path(str(path_or_name)).name.strip()


def normalize_page(page):
    """
    Normaliza páginas para comparar como strings.
    """
    if pd.isna(page):
        return ""
    return str(page).strip()


def get_gold_doc_page_pairs(gold_df, question_id):
    """
    Devuelve todos los pares documento-página válidos para una pregunta.
    """
    rows = gold_df[gold_df["question_id"] == question_id]

    return set(
        (
            normalize_doc_name(row["expected_doc"]),
            normalize_page(row["expected_page"])
        )
        for _, row in rows.iterrows()
    )


def retrieved_to_doc_page_pairs(retrieved_chunks):
    """
    Convierte chunks recuperados en pares documento-página.
    """
    pairs = []

    for chunk in retrieved_chunks:
        doc = normalize_doc_name(chunk.source) if chunk.source else normalize_doc_name(chunk.doc_id)
        page = normalize_page(chunk.page_num)

        pairs.append((doc, page))

    return pairs

def is_relevant_pair(retrieved_pair, gold_pairs, page_tolerance=1):

    retrieved_doc, retrieved_page = retrieved_pair

    try:
        retrieved_page = int(retrieved_page)
    except:
        return False

    for gold_doc, gold_page in gold_pairs:

        try:
            gold_page = int(gold_page)
        except:
            continue

        if (
            retrieved_doc == gold_doc
            and abs(retrieved_page - gold_page) <= page_tolerance
        ):
            return True

    return False

def is_relevant_doc(retrieved_pair, gold_pairs):
    """
    True si el documento recuperado está entre los documentos gold,
    ignorando la página.
    """
    retrieved_doc, _ = retrieved_pair

    return any(
        retrieved_doc == gold_doc
        for gold_doc, _ in gold_pairs
    )


def is_relevant_page(retrieved_pair, gold_pairs, page_tolerance=1):
    """
    True si el documento coincide y la página está dentro de ±page_tolerance.
    """
    retrieved_doc, retrieved_page = retrieved_pair

    try:
        retrieved_page = int(retrieved_page)
    except:
        return False

    for gold_doc, gold_page in gold_pairs:
        try:
            gold_page = int(gold_page)
        except:
            continue

        if retrieved_doc == gold_doc and abs(retrieved_page - gold_page) <= page_tolerance:
            return True

    return False

In [11]:

def chunks_to_context_records(retrieved_chunks):
    context_records = []

    for rank, c in enumerate(retrieved_chunks, start=1):
        context_records.append({
            "rank": rank,
            "doc_id": c.doc_id,
            "chunk_id": c.chunk_id,
            "source": c.source,
            "lang": c.lang,
            "topic": c.topic,
            "distance": c.distance,
            "score": c.score,
            "content": c.content,
        })

    return context_records

def chunks_to_ragas_contexts(retrieved_chunks):
    """
    Formato requerido por Ragas:
    una lista de strings, donde cada string es un contexto/chunk.
    """
    return [
        c.content
        for c in retrieved_chunks
        if c.content is not None and str(c.content).strip()
    ]

## Evaluate semantic search

In [12]:
from app.retrieval import search
import json

def evaluate_search(questions_df, gold_df, page_tolerance=1, TOP_K=5):
    """Evalua un motor de búsqueda para un conjunto de preguntas y referencias gold."""

    retrieval_rows = []

    for _, row in questions_df.iterrows():
        qid = row["question_id"]
        question = row["question"]

        #Obtener los pares de documento-página de referencia para la pregunta actual
        #Devuelve un conjunto de tuplas (doc, page) donde cualquiera de esos documentos/páginas cuenta como evidencia correcta para la pregunta.
        gold_pairs = get_gold_doc_page_pairs(gold_df, qid)

        #Ejecuta el retrieval
        retrieved_chunks = search(question, top_k=TOP_K)

        #Save the retrieved context for debugging and analysis
        retrieved_context = chunks_to_context_records(retrieved_chunks)
        retrieved_context_text = "\n\n".join(
            f"[rank={r['rank']} | {r['doc_id']}:{r['chunk_id']}]\n{r['content']}"
            for r in retrieved_context
        )

        contexts = chunks_to_ragas_contexts(retrieved_chunks)

        # Convierte los chunks recuperados a pares documento-página, el orden se conserva para poder calcular métricas de ranking.
        retrieved_pairs_ranked = retrieved_to_doc_page_pairs(retrieved_chunks)

        #Compara cada documento/página recuperado contra todos los documentos/páginas válidos.
        # LImitación: si hay chunks de la misma página, se puede contar como hit varias veces. 
        doc_hits_by_rank = [
            is_relevant_doc(pair, gold_pairs)
            for pair in retrieved_pairs_ranked
        ]

        page_hits_by_rank = [
            is_relevant_page(pair, gold_pairs, page_tolerance=page_tolerance)
            for pair in retrieved_pairs_ranked
        ]
        
        # Calcular métricas de evaluación
        #Hit@K: ¿Hay al menos un resultado correcto entre los top K?
        doc_hit_at_k = int(any(doc_hits_by_rank))
        page_hit_at_k = int(any(page_hits_by_rank))

        #Identificar en qué posición apareció la primera evidencia correcta.
        first_relevant_doc_rank = None
        for i, is_hit in enumerate(doc_hits_by_rank, start=1):
            if is_hit:
                first_relevant_doc_rank = i
                break

        first_relevant_page_rank = None
        for i, is_hit in enumerate(page_hits_by_rank, start=1):
            if is_hit:
                first_relevant_page_rank = i
                break

        # Calcular el Reciprocal Rank (RR) para la pregunta actual.
        # El RR es 1 dividido por la posición de la primera evidencia correcta. Si no hay evidencia correcta, el RR es 0.
        # Esta métrica premia que la evidencia correcta aparezca arriba.
        doc_reciprocal_rank = (
            0 if first_relevant_doc_rank is None else 1 / first_relevant_doc_rank
        )

        page_reciprocal_rank = (
            0 if first_relevant_page_rank is None else 1 / first_relevant_page_rank
        )

        #Cuenta cuántas evidencias correctas recuperó
        num_relevant_docs_retrieved = sum(doc_hits_by_rank)
        num_relevant_pages_retrieved = sum(page_hits_by_rank)

        #El recall_at_k es básicamente igual que hit_at_k porque estamos usando recall binario.
        # Recall@K: ¿Se recuperó al menos una evidencia correcta entre los top K?
        doc_recall_at_k = int(num_relevant_docs_retrieved > 0)  # binario
        page_recall_at_k = int(num_relevant_pages_retrieved > 0)

        #Precision@K: ¿Qué proporción de los resultados recuperados son correctos? De los K chunks recuperados, ¿cuántos eran documento/página válidos?
        doc_precision_at_k = num_relevant_docs_retrieved / TOP_K
        page_precision_at_k = num_relevant_pages_retrieved / TOP_K

        retrieval_rows.append({
            "question_id": qid,
            "question": question,
            "gold_doc_pages": gold_pairs,
            "retrieved_doc_pages_ranked": retrieved_pairs_ranked,

            "retrieved_context": retrieved_context,
            "contexts": contexts,

            f"doc_hit@{TOP_K}": doc_hit_at_k,
            f"page_hit@{TOP_K}": page_hit_at_k,
            f"doc_precision@{TOP_K}": doc_precision_at_k,
            f"page_precision@{TOP_K}": page_precision_at_k,
            f"doc_binary_recall@{TOP_K}": doc_recall_at_k,
            f"page_binary_recall@{TOP_K}": page_recall_at_k,
            "first_relevant_doc_rank": first_relevant_doc_rank,
            "first_relevant_page_rank": first_relevant_page_rank,
            "doc_reciprocal_rank": doc_reciprocal_rank,
            "page_reciprocal_rank": page_reciprocal_rank,
        })

    retrieval_eval_df = pd.DataFrame(retrieval_rows)

    return retrieval_eval_df

semantic_retrieval_eval_df = evaluate_search(questions_df, gold_df, page_tolerance=page_tolerance, TOP_K=TOP_K)
semantic_retrieval_eval_df.head()

c:\Users\mamen\anaconda3\envs\RAGChatBot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'page_tolerance' is not defined

In [97]:
#print the retrieved context text for question Q001
semantic_retrieval_eval_df[semantic_retrieval_eval_df["question_id"] == "Q001"][["question_id", "contexts"]].iloc[0]["contexts"]

['1. ASTENIA\nAstenia es el término médico para denominar al cansancio (falta de energía) ya\nsea físico o emocional. Al tratarse de un efecto secundario subjetivo, es muy\nimportante que el paciente informe de su presencia. El paciente puede experimentar incapacidad para realizar su actividad física habitual, difi cultad para\nconcentrarse, cansancio que no disminuye pese al descanso…\nTratamiento de la astenia\nLa utilización de fármacos para tratar la astenia es controvertido. Entre los fármacos que han demostrado utilidad son:\n• Corticoides: Producen una mejoría rápida de la astenia. Pueden producir\nademás sensación de euforia y aumento de apetito. Sin embargo, no se recomienda su utilización de forma prolongada por los posibles efectos secundarios a largo plazo como son osteoporosis, atrofi a muscular, síndrome de\nCushing, diabetes corticoidea, infecciones…\n• Progestágenos: Acetato de megestrol.\nMejora el apetito, disminuyendo la sensación de astenia y aumentado el\npeso. El 

In [98]:
def compute_retrieval_metrics(retrieval_eval, TOP_K=5):
    """
    Calcula métricas de evaluación de recuperación a partir del DataFrame de evaluación.
    """
    retrieval_metrics = {

        # Documento
        f"Doc Hit@{TOP_K}":
            retrieval_eval[f"doc_hit@{TOP_K}"].mean(),

        f"Doc Precision@{TOP_K}":
            retrieval_eval[f"doc_precision@{TOP_K}"].mean(),

        f"Doc Binary Recall@{TOP_K}":
            retrieval_eval[f"doc_binary_recall@{TOP_K}"].mean(),

        "Doc MRR":
            retrieval_eval["doc_reciprocal_rank"].mean(),

        # Documento + Página
        f"Page Hit@{TOP_K}":
            retrieval_eval[f"page_hit@{TOP_K}"].mean(),

        f"Page Precision@{TOP_K}":
            retrieval_eval[f"page_precision@{TOP_K}"].mean(),

        f"Page Binary Recall@{TOP_K}":
            retrieval_eval[f"page_binary_recall@{TOP_K}"].mean(),

        "Page MRR":
            retrieval_eval["page_reciprocal_rank"].mean(),
    }

    retrieval_metrics_df = pd.DataFrame(
        retrieval_metrics.items(),
        columns=["metric", "value"]
    )
    return retrieval_metrics_df

semantic_retrieval_metrics_df = compute_retrieval_metrics(semantic_retrieval_eval_df, TOP_K=TOP_K)
semantic_retrieval_metrics_df

,metric,value
0,Doc Hit@5,0.774194
1,Doc Precision@5,0.367742
2,Doc Binary Recall@5,0.774194
3,Doc MRR,0.678495
4,Page Hit@5,0.645161
5,Page Precision@5,0.161290
6,Page Binary Recall@5,0.645161
7,Page MRR,0.477957


In [99]:
semantic_search_eval_path = f"{PROJECT_ROOT}/eval/retrieval_semantic_search_evaluation_results.csv"
semantic_retrieval_eval_df.to_csv( semantic_search_eval_path,
    index=False
)
semantic_retrieval_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_semantic_search_evaluation_metrics.csv", index=False)

In [28]:
semantic_search_eval_path = f"{PROJECT_ROOT}/eval/retrieval_semantic_search_evaluation_results.csv"
semantic_retrieval_eval_df = pd.read_csv(semantic_search_eval_path)
print(f"Total number of questions in semantic_retrieval_eval_df: {len(semantic_retrieval_eval_df)}")
print(f"Columns in semantic_retrieval_eval_df: {semantic_retrieval_eval_df.columns.tolist()}")

#print the context for question Q001
semantic_retrieval_eval_df[semantic_retrieval_eval_df["question_id"] == "Q001"][["question_id", "contexts"]].iloc[0]["contexts"]

Total number of questions in semantic_retrieval_eval_df: 31
Columns in semantic_retrieval_eval_df: ['question_id', 'question', 'gold_doc_pages', 'retrieved_doc_pages_ranked', 'retrieved_context', 'contexts', 'doc_hit@5', 'page_hit@5', 'doc_precision@5', 'page_precision@5', 'doc_binary_recall@5', 'page_binary_recall@5', 'first_relevant_doc_rank', 'first_relevant_page_rank', 'doc_reciprocal_rank', 'page_reciprocal_rank']


"['1. ASTENIA\\nAstenia es el término médico para denominar al cansancio (falta de energía) ya\\nsea físico o emocional. Al tratarse de un efecto secundario subjetivo, es muy\\nimportante que el paciente informe de su presencia. El paciente puede experimentar incapacidad para realizar su actividad física habitual, difi cultad para\\nconcentrarse, cansancio que no disminuye pese al descanso…\\nTratamiento de la astenia\\nLa utilización de fármacos para tratar la astenia es controvertido. Entre los fármacos que han demostrado utilidad son:\\n• Corticoides: Producen una mejoría rápida de la astenia. Pueden producir\\nademás sensación de euforia y aumento de apetito. Sin embargo, no se recomienda su utilización de forma prolongada por los posibles efectos secundarios a largo plazo como son osteoporosis, atrofi a muscular, síndrome de\\nCushing, diabetes corticoidea, infecciones…\\n• Progestágenos: Acetato de megestrol.\\nMejora el apetito, disminuyendo la sensación de astenia y aumentado e

## Evaluate hybrid search

In [41]:
from app.retrieval import hybrid_search

def evaluate_hybrid_search(questions_df, gold_df, page_tolerance=1, TOP_K=5):
    """Evalua un motor de búsqueda híbrido para un conjunto de preguntas y referencias gold."""

    retrieval_rows = []

    for _, row in questions_df.iterrows():
        qid = row["question_id"]
        question = row["question"]

        #Obtener los pares de documento-página de referencia para la pregunta actual
        #Devuelve un conjunto de tuplas (doc, page) donde cualquiera de esos documentos/páginas cuenta como evidencia correcta para la pregunta.
        gold_pairs = get_gold_doc_page_pairs(gold_df, qid)

        #Ejecuta el retrieval
        retrieved_chunks = hybrid_search(question, top_k=TOP_K)
        # Convierte los chunks recuperados a pares documento-página, el orden se conserva para poder calcular métricas de ranking.
        retrieved_pairs_ranked = retrieved_to_doc_page_pairs(retrieved_chunks)

        #Compara cada documento/página recuperado contra todos los documentos/páginas válidos.
        # LImitación: si hay chunks de la misma página, se puede contar como hit varias veces. 
        doc_hits_by_rank = [
            is_relevant_doc(pair, gold_pairs)
            for pair in retrieved_pairs_ranked
        ]

        page_hits_by_rank = [
            is_relevant_page(pair, gold_pairs, page_tolerance=page_tolerance)
            for pair in retrieved_pairs_ranked
        ]
        
        # Calcular métricas de evaluación
        #Hit@K: ¿Hay al menos un resultado correcto entre los top K?
        doc_hit_at_k = int(any(doc_hits_by_rank))
        page_hit_at_k = int(any(page_hits_by_rank))

        #Identificar en qué posición apareció la primera evidencia correcta.
        first_relevant_doc_rank = None
        for i, is_hit in enumerate(doc_hits_by_rank, start=1):
            if is_hit:
                first_relevant_doc_rank = i
                break

        first_relevant_page_rank = None
        for i, is_hit in enumerate(page_hits_by_rank, start=1):
            if is_hit:
                first_relevant_page_rank = i
                break

        # Calcular el Reciprocal Rank (RR) para la pregunta actual.
        # El RR es 1 dividido por la posición de la primera evidencia correcta. Si no hay evidencia correcta, el RR es 0.
        # Esta métrica premia que la evidencia correcta aparezca arriba.
        doc_reciprocal_rank = (
            0 if first_relevant_doc_rank is None else 1 / first_relevant_doc_rank
        )

        page_reciprocal_rank = (
            0 if first_relevant_page_rank is None else 1 / first_relevant_page_rank
        )

        #Cuenta cuántas evidencias correctas recuperó
        num_relevant_docs_retrieved = sum(doc_hits_by_rank)
        num_relevant_pages_retrieved = sum(page_hits_by_rank)

        #El recall_at_k es básicamente igual que hit_at_k porque estamos usando recall binario.
        # Recall@K: ¿Se recuperó al menos una evidencia correcta entre los top K?
        doc_recall_at_k = int(num_relevant_docs_retrieved > 0)  # binario
        page_recall_at_k = int(num_relevant_pages_retrieved > 0)

        #Precision@K: ¿Qué proporción de los resultados recuperados son correctos? De los K chunks recuperados, ¿cuántos eran documento/página válidos?
        doc_precision_at_k = num_relevant_docs_retrieved / TOP_K
        page_precision_at_k = num_relevant_pages_retrieved / TOP_K

        retrieval_rows.append({
        "question_id": qid,
        "question": question,
        "gold_doc_pages": gold_pairs,
        "retrieved_doc_pages_ranked": retrieved_pairs_ranked,

        f"doc_hit@{TOP_K}": doc_hit_at_k,
        f"page_hit@{TOP_K}": page_hit_at_k,

        f"doc_precision@{TOP_K}": doc_precision_at_k,
        f"page_precision@{TOP_K}": page_precision_at_k,

        f"doc_binary_recall@{TOP_K}": doc_recall_at_k,
        f"page_binary_recall@{TOP_K}": page_recall_at_k,

        "first_relevant_doc_rank": first_relevant_doc_rank,
        "first_relevant_page_rank": first_relevant_page_rank,

        "doc_reciprocal_rank": doc_reciprocal_rank,
        "page_reciprocal_rank": page_reciprocal_rank,
        })

    hybrid_retrieval_eval_df = pd.DataFrame(retrieval_rows)
    return hybrid_retrieval_eval_df

hybrid_retrieval_eval_df = evaluate_hybrid_search(questions_df, gold_df, page_tolerance=page_tolerance, TOP_K=TOP_K)

hybrid_retrieval_eval_df.head()

,question_id,question,gold_doc_pages,retrieved_doc_pages_ranked,doc_hit@5,page_hit@5,doc_precision@5,page_precision@5,doc_binary_recall@5,page_binary_recall@5,first_relevant_doc_rank,first_relevant_page_rank,doc_reciprocal_rank,page_reciprocal_rank
0,Q001,¿Qué es la astenia?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 4), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 34), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 177)]",1,1,0.4,0.2,1,1,1.0,1.0,1.0,1.0
1,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 42), (general_es_pfizer_manual-pacientes_2007.pdf, 34), (general_es_pfizer_manual-pacientes_2007.pdf, 150)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 150), (mama_es_esmo_guia-para-pacientes_v1.pdf, 59), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 118), (general_es_pfizer_manual-pacientes_2007.pdf, 71), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 13)]",1,1,0.6,0.2,1,1,1.0,1.0,1.0,1.0
2,Q003,¿Cuál es el trastorno del sueño más común en pacientes con cáncer?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 47)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 48), (mama_es_esmo_guia-para-pacientes_v1.pdf, 38), (prostata_es_esmo_guia-para-pacientes_2022.pdf, 41), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 152), (general_es_pfizer_manual-pacientes_2007.pdf, 72)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0
3,Q004,¿Cómo se define la mucositis?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 175), (mama_es_esmo_guia-para-pacientes_v1.pdf, 49), (mama_es_esmo_guia-para-pacientes_v1.pdf, 43)]",1,1,0.4,0.4,1,1,1.0,1.0,1.0,1.0
4,Q005,¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?,"{(general_es_pfizer_manual-pacientes_2007.pdf, 33)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 33), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 23), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 88), (mama_en_ESMO_early-breast-cancer-guidelines_2023.pdf, 7), (mama_es_esmo_guia-para-pacientes_v1.pdf, 23)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0


In [42]:
hybrid_retrieval_metrics_df = compute_retrieval_metrics(hybrid_retrieval_eval_df, TOP_K=TOP_K)
hybrid_retrieval_metrics_df

,metric,value
0,Doc Hit@5,0.774194
1,Doc Precision@5,0.367742
2,Doc Binary Recall@5,0.774194
3,Doc MRR,0.678495
4,Page Hit@5,0.645161
5,Page Precision@5,0.161290
6,Page Binary Recall@5,0.645161
7,Page MRR,0.477957


```El sistema recuperó al menos un documento relevante para el 77.4% de las preguntas evaluadas. Cuando recuperó documentos relevantes, estos aparecieron generalmente en las primeras posiciones del ranking (MRR=0.678). Considerando además la localización precisa de la información a nivel de página (±1 página de tolerancia), la tasa de éxito fue del 64.5% (Page Hit@5), con un MRR de 0.478. Estos resultados sugieren que el sistema identifica razonablemente bien los documentos pertinentes, aunque existe margen de mejora en la recuperación del fragmento específico que contiene la respuesta.```

In [12]:
hybrid_search_eval_path = f"{PROJECT_ROOT}/eval/retrieval_hybrid_search_evaluation_results.csv"
hybrid_retrieval_eval_df.to_csv( hybrid_search_eval_path,
    index=False
)

hybrid_retrieval_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_hybrid_search_evaluation_metrics.csv", index=False)

## Evaluate language aware hybrid search 

In [43]:
from app.retrieval import language_aware_hybrid_search

def evaluate_language_aware_hybrid_search(questions_df, gold_df, page_tolerance=1, TOP_K=5):
    """Evalua un motor de búsqueda híbrido sensible al lenguaje para un conjunto de preguntas y referencias gold."""

    retrieval_rows = []

    for _, row in questions_df.iterrows():
        qid = row["question_id"]
        question = row["question"]

        #Obtener los pares de documento-página de referencia para la pregunta actual
        #Devuelve un conjunto de tuplas (doc, page) donde cualquiera de esos documentos/páginas cuenta como evidencia correcta para la pregunta.
        gold_pairs = get_gold_doc_page_pairs(gold_df, qid)

        #Ejecuta el retrieval
        retrieved_chunks = language_aware_hybrid_search(question, top_k=TOP_K)
        # Convierte los chunks recuperados a pares documento-página, el orden se conserva para poder calcular métricas de ranking.
        retrieved_pairs_ranked = retrieved_to_doc_page_pairs(retrieved_chunks)

        #Compara cada documento/página recuperado contra todos los documentos/páginas válidos.
        # LImitación: si hay chunks de la misma página, se puede contar como hit varias veces. 
        doc_hits_by_rank = [
            is_relevant_doc(pair, gold_pairs)
            for pair in retrieved_pairs_ranked
        ]

        page_hits_by_rank = [
            is_relevant_page(pair, gold_pairs, page_tolerance=page_tolerance)
            for pair in retrieved_pairs_ranked
        ]
        
        # Calcular métricas de evaluación
        #Hit@K: ¿Hay al menos un resultado correcto entre los top K?
        doc_hit_at_k = int(any(doc_hits_by_rank))
        page_hit_at_k = int(any(page_hits_by_rank))

        #Identificar en qué posición apareció la primera evidencia correcta.
        first_relevant_doc_rank = None
        for i, is_hit in enumerate(doc_hits_by_rank, start=1):
            if is_hit:
                first_relevant_doc_rank = i
                break

        first_relevant_page_rank = None
        for i, is_hit in enumerate(page_hits_by_rank, start=1):
            if is_hit:
                first_relevant_page_rank = i
                break

        # Calcular el Reciprocal Rank (RR) para la pregunta actual.
        # El RR es 1 dividido por la posición de la primera evidencia correcta. Si no hay evidencia correcta, el RR es 0.
        # Esta métrica premia que la evidencia correcta aparezca arriba.
        doc_reciprocal_rank = (
            0 if first_relevant_doc_rank is None else 1 / first_relevant_doc_rank
        )

        page_reciprocal_rank = (
            0 if first_relevant_page_rank is None else 1 / first_relevant_page_rank
        )

        #Cuenta cuántas evidencias correctas recuperó
        num_relevant_docs_retrieved = sum(doc_hits_by_rank)
        num_relevant_pages_retrieved = sum(page_hits_by_rank)

        #El recall_at_k es básicamente igual que hit_at_k porque estamos usando recall binario.
        # Recall@K: ¿Se recuperó al menos una evidencia correcta entre los top K?
        doc_recall_at_k = int(num_relevant_docs_retrieved > 0)  # binario
        page_recall_at_k = int(num_relevant_pages_retrieved > 0)

        #Precision@K: ¿Qué proporción de los resultados recuperados son correctos? De los K chunks recuperados, ¿cuántos eran documento/página válidos?
        doc_precision_at_k = num_relevant_docs_retrieved / TOP_K
        page_precision_at_k = num_relevant_pages_retrieved / TOP_K

        retrieval_rows.append({
        "question_id": qid,
        "question": question,
        "gold_doc_pages": gold_pairs,
        "retrieved_doc_pages_ranked": retrieved_pairs_ranked,

        f"doc_hit@{TOP_K}": doc_hit_at_k,
        f"page_hit@{TOP_K}": page_hit_at_k,

        f"doc_precision@{TOP_K}": doc_precision_at_k,
        f"page_precision@{TOP_K}": page_precision_at_k,

        f"doc_binary_recall@{TOP_K}": doc_recall_at_k,
        f"page_binary_recall@{TOP_K}": page_recall_at_k,

        "first_relevant_doc_rank": first_relevant_doc_rank,
        "first_relevant_page_rank": first_relevant_page_rank,

        "doc_reciprocal_rank": doc_reciprocal_rank,
        "page_reciprocal_rank": page_reciprocal_rank,
        })

    lang_retrieval_eval_df = pd.DataFrame(retrieval_rows)
    return lang_retrieval_eval_df

lang_retrieval_eval_df = evaluate_language_aware_hybrid_search(questions_df, gold_df, page_tolerance=page_tolerance, TOP_K=TOP_K)
lang_retrieval_eval_df.head()

,question_id,question,gold_doc_pages,retrieved_doc_pages_ranked,doc_hit@5,page_hit@5,doc_precision@5,page_precision@5,doc_binary_recall@5,page_binary_recall@5,first_relevant_doc_rank,first_relevant_page_rank,doc_reciprocal_rank,page_reciprocal_rank
0,Q001,¿Qué es la astenia?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 4), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 34), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 177)]",1,1,0.4,0.2,1,1,1.0,1.0,1.0,1.0
1,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 42), (general_es_pfizer_manual-pacientes_2007.pdf, 34), (general_es_pfizer_manual-pacientes_2007.pdf, 150)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 150), (mama_es_esmo_guia-para-pacientes_v1.pdf, 59), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 118), (general_es_pfizer_manual-pacientes_2007.pdf, 71), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 13)]",1,1,0.6,0.2,1,1,1.0,1.0,1.0,1.0
2,Q003,¿Cuál es el trastorno del sueño más común en pacientes con cáncer?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 47)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 48), (mama_es_esmo_guia-para-pacientes_v1.pdf, 38), (prostata_es_esmo_guia-para-pacientes_2022.pdf, 41), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 152), (general_es_pfizer_manual-pacientes_2007.pdf, 72)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0
3,Q004,¿Cómo se define la mucositis?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 175), (mama_es_esmo_guia-para-pacientes_v1.pdf, 49), (mama_es_esmo_guia-para-pacientes_v1.pdf, 43)]",1,1,0.4,0.4,1,1,1.0,1.0,1.0,1.0
4,Q005,¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?,"{(general_es_pfizer_manual-pacientes_2007.pdf, 33)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 33), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 23), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 88), (mama_es_esmo_guia-para-pacientes_v1.pdf, 23), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 36)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0


In [44]:
lang_retrieval_metrics_df = compute_retrieval_metrics(lang_retrieval_eval_df, TOP_K=TOP_K)

lang_retrieval_metrics_df

,metric,value
0,Doc Hit@5,0.774194
1,Doc Precision@5,0.387097
2,Doc Binary Recall@5,0.774194
3,Doc MRR,0.654301
4,Page Hit@5,0.645161
5,Page Precision@5,0.174194
6,Page Binary Recall@5,0.645161
7,Page MRR,0.507527


In [45]:
lang_search_eval_path = f"{PROJECT_ROOT}/eval/retrieval_language_aware_hybrid_search_evaluation_results.csv"
lang_retrieval_eval_df.to_csv( lang_search_eval_path,
    index=False
)

lang_retrieval_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_language_aware_hybrid_search_evaluation_metrics.csv", index=False)

## Combine metrics in a single dataFrame

In [46]:
#Combine retrieval_metrics_df, hybrid_retrieval_metrics_df, lang_retrieval_metrics_df into a single dataframe with an additional column "retrieval_method" indicating the method used.
#The rows will be only 8 of the 8 metrics computed and the values of each method will be added to a column named by the retrieval methdd "semantic_search", "hybrid_search", "language_aware_hybrid_search".

combined_metrics_df = pd.DataFrame({
    "metric": semantic_retrieval_metrics_df["metric"],
    "semantic_search": semantic_retrieval_metrics_df["value"],
    "hybrid_search": hybrid_retrieval_metrics_df["value"],
    "language_aware_hybrid_search": lang_retrieval_metrics_df["value"]
})

combined_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_combined_evaluation_metrics.csv", index=False)
combined_metrics_df

,metric,semantic_search,hybrid_search,language_aware_hybrid_search
0,Doc Hit@5,0.774194,0.774194,0.774194
1,Doc Precision@5,0.367742,0.367742,0.387097
2,Doc Binary Recall@5,0.774194,0.774194,0.774194
3,Doc MRR,0.678495,0.678495,0.654301
4,Page Hit@5,0.645161,0.645161,0.645161
5,Page Precision@5,0.161290,0.161290,0.174194
6,Page Binary Recall@5,0.645161,0.645161,0.645161
7,Page MRR,0.477957,0.477957,0.507527


```
There is no apparent meaningful difference in retrieval coverage between the 3 methods. Language aware and hybrid search may be mostly the same because all the questions are in spanish and most of the documents are also in spanish and that the relevant English documents may not be needed (or as relevant) to answer the benchmark questions.

Equal Doc and Page Hit@5 indicate that the same questions are being solved and missed by all three systems. But Language Aware Hybrid slightly improves precision.

Doc MRR for semantic search and hybrid is higher than language aware search, which could mean that, when looking only at documents, semantic and hybrid search places the first correct document slightly higher in the ranking.

Page MRR for language aware search is higher, which could mean that language aware retrieval finds the correct page slightly earlier than semantic search.
```

De forma orientativa, valores aceptables de retrieval son: 

| Métrica          | Malo | Aceptable | Bueno   | Muy bueno |
| ---------------- | ---- | --------- | ------- | --------- |
| Recall@5 / Hit@5 | <60% | 60-80%    | 80-90%  | >90%      |
| MRR              | <0.4 | 0.4-0.6   | 0.6-0.8 | >0.8      |

Ahora mismo los resultados son buenos/aceptables pero podrían mejorarse más

Opciones para mejorar las métricas:
- Evaluar con mayor tolearancia page_tolearance (2 o 3)
- Mejorar el chunking: 
    - Probar chunks más pequeños (400-500 tokens) (no por página)
    - Mayor overlap (100-150 tokens)
    - Chunking semántico separando por encabezados, párrafos o secciones
- Probar a aumentar el TOP-10 recalcular Hit@10 y MRR@10. Si el Hit sube mucho, el retrieval encuentra información pero demsiado abajo en el ranking.
- Query expansion
- Reranker
- Revisar el modelo de embedding

# Investigate metrics for different TOP_K and page_tolerance values (semantic)

In [17]:
TOP_K = 5
page_tolerance =1

In [ ]:
page_tolerance_values = [0, 1, 2, 3]

combined_tolerance_semantic_metrics_df = pd.DataFrame({
    "metric": semantic_retrieval_metrics_df["metric"],
    "page_tolerance_0": None,
    "page_tolerance_1": None,
    "page_tolerance_2": None,
    "page_tolerance_3": None
})

#Evaluate and compare differences in metrics for different page_tolerance values. For each page_tolerance value, run the evaluate_search function and store the results in a dictionary with the page_tolerance as the key and the resulting dataframe as the value.
for page_tol in page_tolerance_values:
    retrieval_eval_df = evaluate_search(questions_df, gold_df, page_tolerance=page_tol, TOP_K=TOP_K)
    retrieval_metrics_df = compute_retrieval_metrics(retrieval_eval_df, TOP_K=TOP_K)
    combined_tolerance_semantic_metrics_df[f"page_tolerance_{page_tol}"] = retrieval_metrics_df["value"]

combined_tolerance_semantic_metrics_df
    

,metric,page_tolerance_0,page_tolerance_1,page_tolerance_2,page_tolerance_3
0,Doc Hit@5,0.774194,0.774194,0.774194,0.774194
1,Doc Precision@5,0.367742,0.367742,0.367742,0.367742
2,Doc Binary Recall@5,0.774194,0.774194,0.774194,0.774194
3,Doc MRR,0.678495,0.678495,0.678495,0.678495
4,Page Hit@5,0.580645,0.645161,0.677419,0.709677
5,Page Precision@5,0.135484,0.161290,0.200000,0.225806
6,Page Binary Recall@5,0.580645,0.645161,0.677419,0.709677
7,Page MRR,0.423118,0.477957,0.494086,0.526344


In [108]:
combined_tolerance_semantic_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_semantic_search_evaluation_metrics_by_page_tolerance.csv", index=False)

In [24]:
page_tolerance = 1
top_k_values = [1, 3, 5, 10]
metrics_list = ["Doc Hit@K", "Doc Precision@K", "Doc Binary Recall@K", "Doc MRR", "Page Hit@K", "Page Precision@K", "Page Binary Recall@K", "Page MRR"]

combined_tok_k_semantic_metrics_df = pd.DataFrame({
    "metric": metrics_list,
    "top_k_1": None,
    "top_k_3": None,
    "top_k_5": None,
    "top_k_10": None
})

for top_k in top_k_values:
    retrieval_eval_df = evaluate_search(questions_df, gold_df, page_tolerance=page_tolerance, TOP_K=top_k)
    retrieval_metrics_df = compute_retrieval_metrics(retrieval_eval_df, TOP_K=top_k)
    combined_tok_k_semantic_metrics_df[f"top_k_{top_k}"] = retrieval_metrics_df["value"]

combined_tok_k_semantic_metrics_df

,metric,top_k_1,top_k_3,top_k_5,top_k_10
0,Doc Hit@K,0.612903,0.741935,0.774194,0.870968
1,Doc Precision@K,0.612903,0.430108,0.367742,0.338710
2,Doc Binary Recall@K,0.612903,0.741935,0.774194,0.870968
3,Doc MRR,0.612903,0.672043,0.678495,0.690937
4,Page Hit@K,0.387097,0.548387,0.645161,0.741935
5,Page Precision@K,0.387097,0.215054,0.161290,0.122581
6,Page Binary Recall@K,0.387097,0.548387,0.645161,0.741935
7,Page MRR,0.387097,0.456989,0.477957,0.489823


In [109]:
combined_tok_k_semantic_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_semantic_search_evaluation_metrics_by_top_k.csv", index=False)

```
Big difference between Doc and Page Hit = The system often finds the correct document, but not the correct location within that document.
The retrieval is finding relevant information but is not ranking it optimally. 

The large increase from K=1 to K=10 means that relevant information often exists among the retrieved candidates, but it is not appearing at the top K of the ranking.
Difference in Doc and Page MRR = The correct document tends to appear near the top, but the correct chunk/page is often ranked below another chunk from the same document. 
This can mean that the ranking is a bigger problem than the retrieval. If retrieval itself were the main problem, Hit@10 would remain low. 

Would be interesting to analyze failures where page_hit@10 = 0. 

Conclusion 1: top-5 may be unnecessarily restrictive. 
Conclusion 2: A reranker may help more than any other fix. Also improving chunking could help.
Conclusion 3: The retrieval corpus is quite good, the problem may not be the retrieval. 


The decrease in precision is normal. When K increases, recall and hit increase while precision decreases. This is an expected tradeoff. 
```

# Content evaluation (TODO)

In [ ]:
#Save the generated answers to a CSV file in the eval folder.
#Generate the answers using RAG 
from huggingface_hub import InferenceClient
from app.config import settings
from app.rag import rag_answer
from pathlib import Path
import pandas as pd

OUT_PATH = f"{PROJECT_ROOT} /eval/ generated_answers_partial.csv"
output_path = Path(OUT_PATH)
existing = pd.read_csv(output_path) if output_path.exists() else pd.DataFrame()
done_ids = set(existing["question_id"]) if not existing.empty else set()

TOP_K = 5

hf_client = InferenceClient(
    provider="novita",
    api_key=settings.hf_api_key
)

answer_rows = []

for _, row in questions_df.iterrows():
    qid = row["question_id"]

    if qid in done_ids:
        continue

    question = row["question"]

    try:
        answer = rag_answer(
            hf_client=hf_client,
            user_message=question,
            chat_history=[],
            model=settings.llm_model_name,
            top_k=TOP_K,
        )

        new_row = pd.DataFrame([{
            "question_id": qid,
            "question": question,
            "answer": answer,
        }])

        answer_rows.append({
        "question_id": qid,
        "question": question,
        "answer": answer,
        })

        new_row.to_csv(
            output_path,
            mode="a",
            header=not output_path.exists(),
            index=False,
        )

    except Exception as e:
        print(f"Stopped at {qid}: {e}")
        break

answers_df = pd.DataFrame(answer_rows)

answers_df.tail()

In [ ]:
#concatenate the answers_df with the partial_answers_df into a single dataframe with the same columns and save it to a csv file.
print(f"Total number of answers generated: {len(answers_df)}")
print(f"Quesion ids in full_answers_df: {answers_df['question_id'].tolist()}")
full_answers_df_path = f"{PROJECT_ROOT}/eval/full_generated_answers.csv"
answers_df.to_csv(
    full_answers_df_path,
    index=False
)

answers_df.to_json(f"{PROJECT_ROOT}/eval/full_generated_answers.json", orient="records", lines=True)


In [71]:
answers_df.columns

Index(['question_id', 'question', 'answer'], dtype='str')

In [15]:
answers_df_path = f"{PROJECT_ROOT}/eval/full_generated_answers.csv"
answers_df = pd.read_csv(answers_df_path)
print(f"Total number of answers in answers_df: {len(answers_df)}")
print(f"Columns in answers_df: {answers_df.columns.tolist()}")

Total number of answers in answers_df: 31
Columns in answers_df: ['question_id', 'question', 'answer']


## Generate Final DF with the retrieval metrics and context, ground truth and RAG answer

In [29]:
final_df = answers_df.merge(
    semantic_retrieval_eval_df.drop(columns=["question"]),
    on="question_id",
    how="inner"      # o "left" si quieres mantener todas las preguntas de answers_df
)

In [30]:
gold_reference_df = (
    gold_df
    .groupby(["question_id", "question"], as_index=False)
    .agg({
        "expected_text": lambda texts: "\n\n".join(
            str(t).strip() for t in texts if pd.notna(t)
        )
    })
    .rename(columns={"expected_text": "ground_truth"})
)

print(f"Total number of questions in gold_reference_df: {len(gold_reference_df)}")
gold_reference_df.head()

Total number of questions in gold_reference_df: 31


,question_id,question,ground_truth
0,Q001,¿Qué es la astenia?,Astenia es el término médico para denominar al cansancio (falta de energía) ya sea físico o emocional.
1,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,"Si fiebre de más de 38º C, hay que tomar un antipirético (paracetamol, ibuprofeno, metamizol… ) si tras 6-8 horas vuelve a subir la fi ebre, hay que acudir a urgencias para realizar las pruebas pertinentes.\n\nSi tiene temperatura de 38º C o más, debe de ir a urgencias.\nSi tiene fiebre de 38º C o superior, debe avisar a su médico o ir al hospital.\n\nUna temperatura igual o superior a 38ºC indica la posibilidad de una infección y requiere consultar al médico o enfermera qué se debe hacer en una situación así. Hay ocasiones en las que algún medicamento, o la propia enfermedad, pueden provocar fiebre."
2,Q003,¿Cuál es el trastorno del sueño más común en pacientes con cáncer?,El insomnio es el trastorno más común en los pacientes con cáncer y suele ser secundario a factores físicos y psicológicos relacionados con el cáncer y sus tratamientos.
3,Q004,¿Cómo se define la mucositis?,"La mucositis es la inflamación de la mucosa del tracto digestivo que se caracteriza por la aparición de úlceras y/o enrojecimiento, sensación de “quemazón”, etc."
4,Q005,¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?,Suele durar entre 2 y 6 horas.


In [31]:
final_df = final_df.merge(
    gold_reference_df[["question_id", "ground_truth"]],
    on="question_id",
    how="left"
)
print(f"Total number of questions in final_df: {len(final_df)}")
print(f"Columns in final_df: {final_df.columns.tolist()}")
final_df.head(1)

Total number of questions in final_df: 31
Columns in final_df: ['question_id', 'question', 'answer', 'gold_doc_pages', 'retrieved_doc_pages_ranked', 'retrieved_context', 'contexts', 'doc_hit@5', 'page_hit@5', 'doc_precision@5', 'page_precision@5', 'doc_binary_recall@5', 'page_binary_recall@5', 'first_relevant_doc_rank', 'first_relevant_page_rank', 'doc_reciprocal_rank', 'page_reciprocal_rank', 'ground_truth']


question_id             question  \
0        Q001  ¿Qué es la astenia?   

                                                                                                                                                                                                                                                                                     answer  \
0  La astenia es el término médico que describe el cansancio o falta de energía, ya sea física o emocional. Es importante que el paciente informe a su médico sobre la presencia de astenia, ya que puede afectar su capacidad para realizar actividades físicas habituales y concentrarse.   

                                                    gold_doc_pages  \
0  {('general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf', '9')}   

                                                                                                                                                                                                                                                                                                   retrieved_doc_pages_ranked  \
0  [('general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf', '9'), ('prostata_es_gepac_guia-cancer-de-prostata_2020.pdf', '14'), ('general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf', '4'), ('mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf', '34'), ('prostata_es_gepac_guia-cancer-de-prostata_2020.pdf', '177')]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [103]:
final_df.columns

Index(['question_id', 'question', 'answer', 'gold_doc_pages',
       'retrieved_doc_pages_ranked', 'retrieved_context', 'contexts',
       'doc_hit@5', 'page_hit@5', 'doc_precision@5', 'page_precision@5',
       'doc_binary_recall@5', 'page_binary_recall@5',
       'first_relevant_doc_rank', 'first_relevant_page_rank',
       'doc_reciprocal_rank', 'page_reciprocal_rank', 'ground_truth'],
      dtype='str')

In [19]:
final_df_path = f"{PROJECT_ROOT}/eval/final_answers_with_retrieval_eval.csv"
final_df.to_csv(final_df_path, index=False)

## Crear CSV para evaluación manual de completeness y faithfulness

Completeness: ¿La respuesta incluye toda la información importante que debería incluir?

Faithfulness: ¿Todo lo que dice la respuesta está respaldado por el contexto recuperado?

completeness:
- 1 = la respuesta incluye la información esperada
- 0 = la respuesta no incluye la información esperada

faithfulness:
- 1 = la respuesta está respaldada por los documentos/evidencias
- 0 = añade información no soportada o contradice las evidencias

In [8]:
# load final_answers_with_retrieval_eval.csv and check the number of rows and columns.
final_df_path = f"{PROJECT_ROOT}/eval/final_answers_with_retrieval_eval.csv"
final_df = pd.read_csv(final_df_path)
print(f"Total number of rows in final_df: {len(final_df)}")
final_df.head(1)

Total number of rows in final_df: 31


question_id             question  \
0        Q001  ¿Qué es la astenia?   

                                                                                                                                                                                                                                                                                     answer  \
0  La astenia es el término médico que describe el cansancio o falta de energía, ya sea física o emocional. Es importante que el paciente informe a su médico sobre la presencia de astenia, ya que puede afectar su capacidad para realizar actividades físicas habituales y concentrarse.   

                                                    gold_doc_pages  \
0  {('general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf', '9')}   

                                                                                                                                                                                                                                                                                                   retrieved_doc_pages_ranked  \
0  [('general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf', '9'), ('prostata_es_gepac_guia-cancer-de-prostata_2020.pdf', '14'), ('general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf', '4'), ('mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf', '34'), ('prostata_es_gepac_guia-cancer-de-prostata_2020.pdf', '177')]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [ ]:
#merge the answers_df with the semantic_retrieval_eval_df on question_id to create a new dataframe with all the metrics content_eval_df without repeating the question. 
# Add new columns "completeness", "faithfulness", and "notes" to the content_eval_df, all initialized as empty strings. Save the content_eval_df to a CSV file and a JSON file in the eval folder.
"""
content_eval_df = final_df.merge(
    semantic_retrieval_eval_df[[
        "question_id",
        "contexts",
        "ground_truth",
    ]],
    on="question_id",
    how="inner"
)
"""
content_eval_df = final_df[[
    "question_id",
    "question",
    "contexts",
    "ground_truth",
    "answer"
]]

content_eval_df["completeness"] = ""
content_eval_df["faithfulness"] = ""
content_eval_df["notes"] = ""

print(f"Total number of rows in content_eval_df: {len(content_eval_df)}")
content_eval_df.head()

Total number of rows in content_eval_df: 31


question_id  \
0        Q001   
1        Q002   
2        Q003   
3        Q004   
4        Q005   

                                                                    question  \
0                                                        ¿Qué es la astenia?   
1         ¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?   
2         ¿Cuál es el trastorno del sueño más común en pacientes con cáncer?   
3                                              ¿Cómo se define la mucositis?   
4  ¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [11]:
content_evaluation_manual_path = f"{PROJECT_ROOT}/eval/content_evaluation_manual.csv"
content_eval_df.to_csv( content_evaluation_manual_path,index=False)
content_eval_df.to_json(f"{PROJECT_ROOT}/eval/content_evaluation_manual.json", orient="records", lines=True)

In [7]:
# Cargar el archivo CSV generado y verificar que se haya guardado correctamente.
content_evaluation_manual_path = f"{PROJECT_ROOT}/eval/content_evaluation_manual.csv"
loaded_content_eval_df = pd.read_csv(content_evaluation_manual_path)
print(f"Total number of rows in loaded_content_eval_df: {len(loaded_content_eval_df)}")
loaded_content_eval_df.head(1)

Total number of rows in loaded_content_eval_df: 12


question_id  \
0        Q020   

                                                                       question  \
0  ¿Qué recomendaciones específicas hay para pacientes tratadas con tamoxifeno?   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [5]:
# save the loaded_content_eval_df to a json file in the eval folder.
loaded_content_eval_df.to_json(f"{PROJECT_ROOT}/eval/content_evaluation_manual.json", orient="records", lines=True)

## Cargar el content evaluation

Completeness
| Valor | Interpretación                                   |
| ----: | ------------------------------------------------ |
|  1.00 | Contiene todos los puntos necesarios             |
|  0.75 | Contiene la mayoría; faltan detalles secundarios |
|  0.50 | Cubre aproximadamente la mitad                   |
|  0.25 | Solo incluye una parte pequeña                   |
|  0.00 | No proporciona la información solicitada         |


Faithfulness
| Valor | Interpretación                                                |
| ----: | ------------------------------------------------------------- |
|  1.00 | Todas las afirmaciones están respaldadas                      |
|  0.75 | Contenido principalmente respaldado, con alguna adición menor |
|  0.50 | Mezcla similar de afirmaciones respaldadas y no respaldadas   |
|  0.25 | La mayoría no está respaldada o contradice las evidencias     |
|  0.00 | La respuesta es completamente incompatible con las evidencias |


In [46]:
content_evaluation_path = f"{PROJECT_ROOT}/eval/content_evaluation_full.json"
content_eval_df = pd.read_json(content_evaluation_path, orient="records", lines=True)

#Ajustes
#asignar el completeness de la pregunta Q002 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q002", "completeness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q004", "faithfulness"] = 1
#content_eval_df.loc[content_eval_df["question_id"] == "Q004", "notes"] += "Se ha comprobado que la clasificación por grados sí se incluye en el contexto recuperado, por lo que la respuesta es fiel a la evidencia. "
content_eval_df.loc[content_eval_df["question_id"] == "Q006", "faithfulness"] = 1
#content_eval_df.loc[content_eval_df["question_id"] == "Q006", "notes"] += "Se ha comprobado que información extra sí se incluye en el contexto recuperado, por lo que la respuesta es fiel a la evidencia. "
content_eval_df.loc[content_eval_df["question_id"] == "Q008", "completeness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q013", "completeness"] = 0.50
content_eval_df.loc[content_eval_df["question_id"] == "Q013", "faithfulness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q014", "completeness"] = 0.65
content_eval_df.loc[content_eval_df["question_id"] == "Q014", "faithfulness"] = 1
content_eval_df.loc[content_eval_df["question_id"] == "Q014", "notes"] = "Completitud: la respuesta identifica correctamente que radio-223 es un radiofármaco intravenoso utilizado para tratar el cáncer de próstata extendido a los huesos y explica que se deposita en las lesiones óseas, donde su radiación elimina células cancerosas. Sin embargo, omite beneficios relevantes descritos en el contexto, como el alivio del dolor, la prolongación de la supervivencia, el retraso de eventos esqueléticos y la mejora de la calidad de vida, así como condiciones y precauciones específicas de uso. Fidelidad: todas las afirmaciones incluidas están directamente respaldadas por el contexto, por lo que no se detecta información inventada o contradictoria."
content_eval_df.loc[content_eval_df["question_id"] == "Q015", "completeness"] = 0.25
content_eval_df.loc[content_eval_df["question_id"] == "Q015", "faithfulness"] = 0.5
content_eval_df.loc[content_eval_df["question_id"] == "Q019", "faithfulness"] = 0.5
content_eval_df.loc[content_eval_df["question_id"] == "Q020", "completeness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q020", "faithfulness"] = 1
content_eval_df.loc[content_eval_df["question_id"] == "Q020", "notes"] = "Completitud: la respuesta recoge correctamente la anticoncepción no hormonal, el seguimiento ginecológico periódico, la derivación al cardiólogo ante insuficiencia cardiaca después de quimioterapia (no presente en el ground_truth pero sí en el contexto recuperado) y la suspensión del tamoxifeno con valoración oncológica ante una TVP. Sin embargo, omite la posible interacción de determinados medicamentos con el metabolismo del tamoxifeno mediante CYP2D6, especialmente algunos antidepresivos, que puede disminuir su eficacia. También omite algunos detalles secundarios sobre la secuencia con quimioterapia y la posible prolongación del tratamiento en pacientes seleccionadas. Fidelidad: todas las recomendaciones incluidas están respaldadas directamente por los contextos recuperados (aunque no totalmente por la evaluación obtenida manualmente), sin afirmaciones contradictorias o no sustentadas."
content_eval_df.loc[content_eval_df["question_id"] == "Q021", "completeness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q021", "notes"] = "Completitud: la respuesta incluye casi todos los síntomas relevantes presentes en los contextos: sequedad vaginal, sofocos, insomnio, dispareunia, reducción de la lubricación, pérdida del deseo sexual y síntomas urinarios como disuria, aumento de la frecuencia, urgencia y cistitis poscoital. Omite el picor vulvovaginal y repite algunos conceptos, especialmente la sequedad vaginal y la dispareunia. Algunos de estos síntomas no están contenidos en el ground truth pero eso se considera una limitación de la búsqueda manual y no del sistema. Fidelidad: todos los síntomas enumerados aparecen en los contextos recuperados. No obstante, la respuesta los atribuye globalmente a la falta de estrógenos, mientras que algunos fragmentos los presentan específicamente como síntomas de menopausia o efectos de los inhibidores de la aromatasa; por ello, la atribución causal es razonable, pero ligeramente más general que la evidencia recuperada."
content_eval_df.loc[content_eval_df["question_id"] == "Q023", "completeness"] = 0.25
content_eval_df.loc[content_eval_df["question_id"] == "Q023", "faithfulness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q023", "notes"] = "Completitud: la respuesta identifica correctamente que la quimioterapia puede provocar supresión o insuficiencia ovárica y reducir la producción de estrógenos. Sin embargo, omite las frecuencias por grupos de edad, la dependencia del riesgo respecto al tipo y dosis de quimioterapia y la edad de la paciente, el mayor riesgo en mujeres de más edad y la necesidad de informar sobre criopreservación de ovocitos antes del tratamiento. Tampoco menciona las consecuencias sobre la fertilidad y los síntomas menopáusicos. Fidelidad: la supresión ovárica producida por la quimioterapia está respaldada por el ground truth y los contextos. No obstante, la respuesta mezcla este efecto adverso con la supresión ovárica terapéutica y afirma sin respaldo que la quimioterapia puede ser menos eficaz en mujeres posmenopáusicas debido a sus menores niveles de estrógenos, confundiendo la quimioterapia con la terapia endocrina."
content_eval_df.loc[content_eval_df["question_id"] == "Q024", "faithfulness"] = 0.5
content_eval_df.loc[content_eval_df["question_id"] == "Q024", "notes"] = "Completitud: la respuesta define correctamente la dispareunia como dolor o incomodidad durante las relaciones sexuales, pero omite que el dolor o molestia puede aparecer antes, durante o después de la unión sexual, tal como especifica el ground truth. Fidelidad: la definición inicial está respaldada por el ground truth. Sin embargo, los contextos recuperados no relacionan directamente la dispareunia con problemas urinarios, cirugía prostática o tratamientos para el cáncer de próstata, ni indican que sea un problema común en estos pacientes. Además, afirmar que la dispareunia no se menciona en los textos es incompatible con el ground truth, que sí proporciona una definición explícita."
content_eval_df.loc[content_eval_df["question_id"] == "Q025", "completeness"] = 0.5
content_eval_df.loc[content_eval_df["question_id"] == "Q025", "faithfulness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q025", "notes"] = "Completitud: la respuesta confirma correctamente que es posible quedarse embarazada después del tratamiento y recoge que la terapia endocrina puede interrumpirse temporalmente, bajo control médico y durante un máximo de dos años, sin aumento demostrado de la recaída a corto plazo. Sin embargo, omite que generalmente se recomienda esperar al menos dos años después del tratamiento, que el momento depende del estadio y del tratamiento recibido, y que la terapia endocrina debe suspenderse antes de intentar el embarazo y reanudarse después del parto y la lactancia. Fidelidad: la posibilidad de embarazo, la interrupción temporal de la terapia endocrina y la recomendación de consultar al médico están respaldadas. No obstante, la referencia a haber superado el primer trimestre procede de un contexto sobre la realización segura de la biopsia selectiva del ganglio centinela en pacientes ya embarazadas y no constituye una condición para quedarse embarazada después del cáncer; por tanto, es una aplicación incorrecta de información recuperada."
content_eval_df.loc[content_eval_df["question_id"] == "Q026", "completeness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q026", "faithfulness"] = 1
content_eval_df.loc[content_eval_df["question_id"] == "Q026", "notes"] = "Completitud: la respuesta recoge correctamente numerosos factores de riesgo presentes en los contextos y el ground truth, incluidos sexo femenino, edad avanzada, radiaciones ionizantes, predisposición genética y antecedentes familiares, pocos hijos, hiperplasia atípica, obesidad, exposición a estrógenos y alcohol. Sin embargo, omite factores adicionales relevantes del ground truth, como antecedentes personales de cáncer de mama o carcinoma ductal in situ, densidad mamaria elevada, menarquia temprana o menopausia tardía, nuliparidad, primer embarazo tardío, ausencia de lactancia, terapia hormonal sustitutiva, anticonceptivos orales, sedentarismo y dieta inadecuada. Fidelidad: todos los factores mencionados y la aclaración sobre el carácter probabilístico de los factores de riesgo están directamente respaldados por los contexts y el ground truth; no se identifican afirmaciones inventadas o contradictorias."

content_eval_df.loc[content_eval_df["question_id"] == "Q027", "completeness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q027", "faithfulness"] = 0.75
content_eval_df.loc[content_eval_df["question_id"] == "Q027", "notes"] = "Completitud: la respuesta recoge correctamente el uso de la petición electrónica mediante Diraya y PDI, la necesidad de incluir la justificación clínica, los tres niveles de prioridad y la fecha ideal de realización. También explica que el Servicio de Radiodiagnóstico valorará la pertinencia, modalidad y prioridad del estudio. Sin embargo, omite que en los controles periódicos debe indicarse la fecha de la siguiente revisión en consulta y no especifica que el procedimiento se aplica tanto a Atención Primaria como a Atención Especializada. Fidelidad: la mayor parte de la respuesta está respaldada, pero atribuye al solicitante la obligación de especificar mamografía o ecografía. Según el contexto, es Radiodiagnóstico quien decide la prueba inicial después de valorar la clínica, edad y antecedentes de la paciente. Esta atribución incorrecta reduce la fidelidad."

content_eval_df.loc[content_eval_df["question_id"] == "Q028", "completeness"] = 0.5
content_eval_df.loc[content_eval_df["question_id"] == "Q028", "faithfulness"] = 0.75

content_eval_df.loc[content_eval_df["question_id"] == "Q031", "faithfulness"] = 0.25

content_eval_df[["question_id", "question", "ground_truth", "answer", "completeness", "faithfulness", "notes"]].tail(7)

,question_id,question,ground_truth,answer,completeness,faithfulness,notes
24,Q025,¿Puedo quedarme embarazada después de un tratamiento de cancer?,"Available data suggest that pregnancy is safe after breast cancer treatment\n\nNo existen contraindicaciones para quedar embarazada después de tener cáncer de mama. Sin embargo, hay varios puntos importantes a considerar, especialmente en el caso de cáncer de mama RE positivo, debido a la larga duración de la terapia endocrina. La terapia endocrina debe suspenderse antes de tratar de quedar embarazada y debe reanudarse después del parto y la lactancia. Si usted desea quedar embarazada, por favor discuta todas estas cuestiones cuidadosamente con su médico.\n\nLa mayoría de las guías recomiendan posponer el embarazo duran te al menos 2 años después del tratamiento del cáncer de mama, según el estadio y el tratamiento recibido. Sin embargo, es poco probable que las mujeres con enfermedad localizada empeoren el pronóstico si se quedan embarazadas. Estudios recientes evidencian que, en pacientes con cáncer de mama, pausar la terapia endocrina para poder quedarse embara zadas —bajo control médico y durante un periodo temporal que no supere los 2 años— no incrementa el riesgo de recaída a corto pla zo.","Sí, es posible quedarte embarazada después de un tratamiento de cáncer de mama, especialmente si has superado el primer trimestre de embarazo. De hecho, algunos estudios recientes muestran que pausar la terapia endocrina para poder quedarte embarazada, bajo control médico y durante un período temporal que no supere los 2 años, no incrementa el riesgo de recaída a corto plazo.\n\nEs importante que consultes con tu médico para conocer tu situación específica y obtener consejos personalizados sobre cómo manejar tu embarazo después del tratamiento de cáncer.",0.50,0.75,"Completitud: la respuesta confirma correctamente que es posible quedarse embarazada después del tratamiento y recoge que la terapia endocrina puede interrumpirse temporalmente, bajo control médico y durante un máximo de dos años, sin aumento demostrado de la recaída a corto plazo. Sin embargo, omite que generalmente se recomienda esperar al menos dos años después del tratamiento, que el momento depende del estadio y del tratamiento recibido, y que la terapia endocrina debe suspenderse antes de intentar el embarazo y reanudarse después del parto y la lactancia. Fidelidad: la posibilidad de embarazo, la interrupción temporal de la terapia endocrina y la recomendación de consultar al médico están respaldadas. No obstante, la referencia a haber superado el primer trimestre procede de un contexto sobre la realización segura de la biopsia selectiva del ganglio centinela en pacientes ya embarazadas y no constituye una condición para quedarse embarazada después del cáncer; por tanto, es una aplicación incorrecta de información recuperada."
25,Q026,¿cuáles son los factores de riesgo del cancer de mama?,"El cáncer de mama tiene un origen multifactorial, y, en la mayoría de los casos, no se puede identificar de forma clara la causa. Sí conocemos sin embargo muchos factores de riesgo asociados al cáncer de mama, que pueden dividirse en factores de riesgo modificables o no modificables. La mayor parte de los ellos se relaciona con los antecedentes reproductivos que modulan la exposición hormonal durante la vida.\n1. La edad es el principal factor de riesgo para padecer un cáncer de mama. El riesgo aumenta al aumentar la edad. 2. Historia personal de cáncer de mama invasivo (las mujeres que han tenido un cáncer de mama invasivo tienen más riesgo de padecer un cáncer de mama contralateral), carcinoma ductal in situ o lesiones benignas como la hiperplasia epitelial atípica. 3. Densidad mamaria elevada en las mamografías. 4. Historia menstrual larga, con una menarquia (edad de primera regla) temprana o una menopausia tardía 5. Exposición a radiaciones ionizantes, sobre todo durante la pubertad 6. Factores genéticos como mutaciones hereditarias 

In [37]:
content_eval_df.loc[content_eval_df["question_id"] == "Q020",:]

question_id  \
19        Q020   

                                                                        question  \
19  ¿Qué recomendaciones específicas hay para pacientes tratadas con tamoxifeno?   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [48]:
#Guardar el DataFrame actualizado en un archivo CSV y JSON en la carpeta eval.
content_eval_df.to_csv(f"{PROJECT_ROOT}/eval/content_evaluation_full_updated.csv", index=False)
content_eval_df.to_json(f"{PROJECT_ROOT}/eval/content_evaluation_full_updated.json", orient="records", lines=True)

# Metrica combinada: porcentaje de preguntas donde retrieval acertó, la respuesta fue completa y además fiel.

In [ ]:
#Crear un resumen de las métricas agregadas de completeness y faithfulness, calculando la media, mediana, desviación estándar, mínimo y máximo para cada métrica. Guardar este resumen en un archivo CSV y JSON en la carpeta eval.
summary_metrics_df = content_eval_df[["completeness", "faithfulness"]].agg(["mean", "median", "std", "min", "max"]).reset_index().rename(columns={"index": "metric"})
print(f"Summary metrics:\n{summary_metrics_df}")
summary_metrics_df.to_csv(f"{PROJECT_ROOT}/eval/content_evaluation_summary_metrics.csv", index=False)
summary_metrics_df.to_json(f"{PROJECT_ROOT}/eval/content_evaluation_summary_metrics.json", orient="records", lines=True)

Summary metrics:
   metric  completeness  faithfulness
0    mean      0.608065      0.854839
1  median      0.750000      1.000000
2     std      0.364256      0.211878
3     min      0.000000      0.250000
4     max      1.000000      1.000000


In [ ]:
"""Crear un archivo MD con el summary metrics, notas sobre las observaciones hechas durante la evaluación del contenido y explicación de qué es completeness y faithfulness. 
Añadir además las siguientes tablas explicando los valores dados.

Completeness: ¿La respuesta incluye toda la información importante que debería incluir?
Faithfulness: ¿Todo lo que dice la respuesta está respaldado por el contexto recuperado?

Completeness
| Valor | Interpretación                                   |
| ----: | ------------------------------------------------ |
|  1.00 | Contiene todos los puntos necesarios             |
|  0.75 | Contiene la mayoría; faltan detalles secundarios |
|  0.50 | Cubre aproximadamente la mitad                   |
|  0.25 | Solo incluye una parte pequeña                   |
|  0.00 | No proporciona la información solicitada         |


Faithfulness
| Valor | Interpretación                                                |
| ----: | ------------------------------------------------------------- |
|  1.00 | Todas las afirmaciones están respaldadas                      |
|  0.75 | Contenido principalmente respaldado, con alguna adición menor |
|  0.50 | Mezcla similar de afirmaciones respaldadas y no respaldadas   |
|  0.25 | La mayoría no está respaldada o contradice las evidencias     |
|  0.00 | La respuesta es completamente incompatible con las evidencias |

Número total de preguntas evaluadas: 31
Summary metrics:
   metric  completeness  faithfulness
0    mean      0.608065      0.854839
1  median      0.750000      1.000000
2     std      0.364256      0.211878
3     min      0.000000      0.250000
4     max      1.000000      1.000000
"""

md_content = f"""# Evaluación de Contenido Generado por RAG
## Definición de Métricas
### Completeness

Completeness evalúa si la respuesta generada incluye toda la información importante que debería incluir según el contexto recuperado. Una respuesta completa proporciona una cobertura adecuada de los puntos clave y detalles relevantes, mientras que una respuesta incompleta puede omitir información crítica o relevante.

| Valor | Interpretación                                   |
| ----: | ------------------------------------------------ |    
|  1.00 | Contiene todos los puntos necesarios             |
|  0.75 | Contiene la mayoría; faltan detalles secundarios |
|  0.50 | Cubre aproximadamente la mitad                   |
|  0.25 | Solo incluye una parte pequeña                   |
|  0.00 | No proporciona la información solicitada         |
### Faithfulness

Faithfulness evalúa si todo lo que dice la respuesta está respaldado por el contexto recuperado. Una respuesta fiel refleja con precisión la información contenida en los documentos de referencia, mientras que una respuesta infiel puede incluir afirmaciones no respaldadas o contradictorias con la evidencia disponible.

| Valor | Interpretación                                                |
| ----: | ------------------------------------------------------------- |   
|  1.00 | Todas las afirmaciones están respaldadas                      |
|  0.75 | Contenido principalmente respaldado, con alguna adición menor |
|  0.50 | Mezcla similar de afirmaciones respaldadas y no respaldadas   |
|  0.25 | La mayoría no está respaldada o contradice las evidencias     |
|  0.00 | La respuesta es completamente incompatible con las evidencias |

## Número Total de Preguntas Evaluadas: 31

## Resumen de Métricas  

| Métrica      | Media  | Mediana | Desviación Estándar | Mínimo | Máximo |
| ------------ | ------ | ------- | ------------------ | ------ | ------ |
| Completeness | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'mean', 'completeness'].values[0]:.2f} | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'median', 'completeness'].values[0]:.2f} | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'std', 'completeness'].values[0]:.2f} | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'min', 'completeness'].values[0]:.2f} | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'max', 'completeness'].values[0]:.2f} |
| Faithfulness | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'mean', 'faithfulness'].values[0]:.2f} | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'median', 'faithfulness'].values[0]:.2f} | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'std', 'faithfulness'].values[0]:.2f} | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'min', 'faithfulness'].values[0]:.2f} | {summary_metrics_df.loc[summary_metrics_df['metric'] == 'max', 'faithfulness'].values[0]:.2f} |    

## Observaciones de la Evaluación de Contenido
Durante la evaluación se observó que había varios casos en los que la respuesta generada decía que no existía contexto que respondiera a dicha pregunta (Cuando la evidencia si responde a dicha pregunta). En estos casos la completitud se definió como 0 y la fiabilidad como 1 ya que el sistema reconoció de manera adecuada que no sabía responder a la preguntas.

También hay varias ocasiones en las que se observó que el ground_truth no recogía toda la información relevante que sí estaba presente en los contextos recuperados. En estos casos, se tuvo en cuenta también la información recuperada por el sistema de retrieval para la evaluación. 
Se reconoce que esto es una limitación de la selección manual de las respuestas más que una limitación del sistema de RAG. 

También se pudo observar que aquellas respuestas donde la completitud era baja, se debía en muchos casos a que el sistema de retrieval no había recuperado toda la información relevante para responder a la pregunta. Esto indica que el sistema de retrieval es un componente crítico para la generación de respuestas completas y precisas.
Aún así, en la mayoría de casos, la respuesta se mantiene fiel a la evidencia recuperada, lo que indica que el sistema de generación de respuestas es capaz de utilizar la información disponible de manera efectiva.
"""

#Guardar el contenido en un archivo Markdown en la carpeta eval. Preservar correctamente las tildes del tetxo.
md_file_path = f"{PROJECT_ROOT}/eval/content_evaluation_summary.md"
with open(md_file_path, "w", encoding="utf-8") as md_file:
    md_file.write(md_content)



# Resumen de metricas

In [ ]:
final_metrics = {
    f"Hit@{TOP_K}": manual_eval_df[f"hit@{TOP_K}"].mean(),
    f"Precision@{TOP_K}": manual_eval_df[f"precision@{TOP_K}"].mean(),
    f"Binary Recall@{TOP_K}": manual_eval_df[f"binary_recall@{TOP_K}"].mean(),
    "MRR": manual_eval_df["reciprocal_rank"].mean(),
    "Completeness": manual_eval_df["completeness"].mean(),
    "Faithfulness": manual_eval_df["faithfulness"].mean(),
    "End-to-end success": manual_eval_df["end_to_end_success"].mean(),
}

final_metrics_df = pd.DataFrame(
    final_metrics.items(),
    columns=["metric", "value"]
)

final_metrics_df

In [ ]:
final_metrics_df.to_csv(
    PROJECT_ROOT / "rag_evaluation_metrics_summary.csv",
    index=False
)

manual_eval_df.to_csv(
    PROJECT_ROOT / "rag_evaluation_full_results.csv",
    index=False
)

# Usar el Framework RAGAS para evaluar el sistema RAG

`Sirve para evaluar de manera automática el sistema pero requiere una API KEY de OPENAI o el uso de algún LLM alternativo para evaluar las respuestas`

- https://medium.com/@sanjeebmeister/rag-evaluation-metrics-explained-a-complete-guide-with-examples-dea8bf4467db


Metricas calculadas por RAGAS

| Métrica              | Qué mide                                                                             | Usa ground truth        |
| -------------------- | ------------------------------------------------------------------------------------ | ----------------------- |
| `faithfulness`       | Si la respuesta está respaldada por el contexto recuperado                           | No                      |
| `answer_relevancy`   | Si la respuesta responde a la pregunta                                               | No                      |
| `context_precision`  | Si los contextos relevantes aparecen bien posicionados                               | Sí o depende de versión |
| `context_recall`     | Si el contexto recuperado contiene la información necesaria respecto a la referencia | Sí                      |
| `answer_correctness` | Similitud factual/semántica entre respuesta y ground truth                           | Sí                      |


Tu ground_truth viene de concatenar evidencias documentales, no de una respuesta clínica redactada. Por tanto:
- faithfulness es muy fiable para detectar alucinaciones.
- answer_relevancy es útil para saber si responde a la pregunta.
- context_recall es útil si tu ground_truth representa bien toda la información esperada.
- answer_correctness úsala como métrica complementaria, no como verdad absoluta, porque puede penalizar respuestas correctas pero más resumidas que tus evidencias.

In [20]:
#load final_df
final_df = pd.read_csv(f"{PROJECT_ROOT}/eval/final_answers_with_retrieval_eval.csv")
print(f"Total number of questions in final_df: {len(final_df)}")
print(f"Columns in final_df: {final_df.columns.tolist()}")

Total number of questions in final_df: 31
Columns in final_df: ['question_id', 'question', 'answer', 'gold_doc_pages', 'retrieved_doc_pages_ranked', 'retrieved_context', 'contexts', 'doc_hit@5', 'page_hit@5', 'doc_precision@5', 'page_precision@5', 'doc_binary_recall@5', 'page_binary_recall@5', 'first_relevant_doc_rank', 'first_relevant_page_rank', 'doc_reciprocal_rank', 'page_reciprocal_rank', 'ground_truth']


In [53]:
from dotenv import load_dotenv
import os

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
embedding_model = os.getenv("EMBED_MODEL_NAME")

print(os.getenv("OPENAI_API_KEY") is not None)
print(os.getenv("EMBED_MODEL_NAME"))

True
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [44]:
import ast
import pandas as pd
from datasets import Dataset

def ensure_list_of_strings(value):
    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]

    if pd.isna(value):
        return []

    value = str(value).strip()

    # Caso: "['ctx1', 'ctx2']"
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if str(x).strip()]
        except Exception:
            pass

    # Caso: texto único
    return [value]

In [45]:
ragas_df = final_df.copy()

ragas_df["contexts"] = ragas_df["contexts"].apply(ensure_list_of_strings)

ragas_input_df = ragas_df[
    ["question", "answer", "contexts", "ground_truth"]
].copy()

ragas_input_df = ragas_input_df.rename(
    columns={
        "question": "user_input",
        "answer": "response",
        "ground_truth": "reference",
    }
)

dataset = Dataset.from_pandas(ragas_input_df, preserve_index=False)

In [46]:
# Comprobar una fila

i = 0

print("QUESTION:")
print(ragas_input_df.loc[i, "user_input"])

print("\nANSWER:")
print(ragas_input_df.loc[i, "response"])

print("\nGROUND TRUTH:")
print(ragas_input_df.loc[i, "reference"])

print("\nCONTEXTS:")
print(ragas_input_df.loc[i, "contexts"])

QUESTION:
¿Qué es la astenia?

ANSWER:
La astenia es el término médico que describe el cansancio o falta de energía, ya sea física o emocional. Es importante que el paciente informe a su médico sobre la presencia de astenia, ya que puede afectar su capacidad para realizar actividades físicas habituales y concentrarse.

GROUND TRUTH:
Astenia es el término médico para denominar al cansancio (falta de energía) ya sea físico o emocional.

CONTEXTS:
['1. ASTENIA\nAstenia es el término médico para denominar al cansancio (falta de energía) ya\nsea físico o emocional. Al tratarse de un efecto secundario subjetivo, es muy\nimportante que el paciente informe de su presencia. El paciente puede experimentar incapacidad para realizar su actividad física habitual, difi cultad para\nconcentrarse, cansancio que no disminuye pese al descanso…\nTratamiento de la astenia\nLa utilización de fármacos para tratar la astenia es controvertido. Entre los fármacos que han demostrado utilidad son:\n• Corticoides

In [56]:
import os
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

results = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
        answer_correctness,
    ],
    llm=llm,
    embeddings=embeddings,
)

results_df = results.to_pandas()
results_df.head()

C:\Users\mamen\AppData\Local\Temp\ipykernel_6796\3793064699.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Evaluating:   1%|          | 1/155 [00:00<01:14,  2.06it/s]Exception raised in Job[2]: AuthenticationError(Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************aiYA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401})
E

user_input  \
0                                                        ¿Qué es la astenia?   
1         ¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?   
2         ¿Cuál es el trastorno del sueño más común en pacientes con cáncer?   
3                                              ¿Cómo se define la mucositis?   
4  ¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [ ]:
#Añadir el question_id al dataframe de resultados de Ragas para poder unirlo con el dataframe original.
ragas_results_df = pd.concat(
    [
        ragas_df.loc[ragas_input_df.index, ["question_id"]].reset_index(drop=True),
        ragas_results_df.reset_index(drop=True),
    ],
    axis=1
)

ragas_results_df.head()

In [ ]:
#Calcular el promedio de cada métrica de Ragas para todas las preguntas y guardarlo en un nuevo dataframe.
ragas_metric_cols = [
    col for col in ragas_results_df.columns
    if col != "question_id"
    and pd.api.types.is_numeric_dtype(ragas_results_df[col])
]

ragas_summary_df = (
    ragas_results_df[ragas_metric_cols]
    .mean()
    .reset_index()
)

ragas_summary_df.columns = ["metric", "mean_score"]

ragas_summary_df

In [ ]:
#Guardar los resultados de Ragas en un archivo CSV y JSON en la carpeta eval.
ragas_results_df.to_csv("ragas_results.csv", index=False)
ragas_summary_df.to_csv("ragas_summary.csv", index=False)